This code builds on the CAR_data_import code given by Krish

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
df = pd.read_csv('total_call_data.csv')

C:\Users\Owner\AppData\Local\Temp\ipykernel_19860\31442611.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('total_call_data.csv')


In [3]:
df.columns

Index(['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name',
       'Activity Start Timestamp', 'Queue Name', 'Agent Name',
       'Termination Reason', 'hour'],
      dtype='object')

In [4]:
df['Contact Session ID'].nunique()

240592

In [6]:
# Example target activity names
target_activities = ["Queue"]

# Assuming your DataFrame is named df and has at least these columns:
# "Call ID" (or whatever uniquely identifies a call) and "Activity Name"

# Step 1: Find all call IDs that hit one of the target activities
call_ids_to_keep = df.loc[(df["Queue Name"].isna() == False) & 
                          (df['Queue Name'] != 'Clinic Voicemail Transfer') &
                          (df['Queue Name'] != 'Intake Outdial Queue') &
                          (df['Queue Name'] != 'Front Desk Transfer') &
                          (df['Queue Name'] != 'Staff Directory English Transfer') &
                          (df['Queue Name'] != 'Criminal Records Voicemail Transfer') &
                          (df['Queue Name'] != 'Staff Directory Spanish Transfer') &
                          (df['Queue Name'] != 'HIV Voicemail Transfer') &
                          (df['Queue Name'] != 'Trafficking Voicemail Transfer') &
                          (df['Queue Name'] != 'Veterans Benefits Voicemail Transfer') &
                          (df['Queue Name'] != 'Farmworker Voicemail Transfer') &
                          (df['Queue Name'] != 'Safe Haven Transfer'), "Contact Session ID"].unique()

# Step 2: Filter to keep *all* rows of those calls
filtered_df = df[df["Contact Session ID"].isin(call_ids_to_keep)].copy().reset_index(drop = True)

In [10]:
#### These are all the Queue Names found outside of the main Queue activity
# Intake Outdial Queue (figure out what that is )
# Clinic Voicemail Transfer
# Front Desk Transfer
# Staff Directory English Transfer
# Criminal Records Voicemail Transfer
# HIV Voicemail Transfer
# Trafficking Voicemail Transfer
# Staff Directory Spanish Transfer
# Veterans Benefits Voicemail Transfer
# Farmworker Voicemail Transfer
# Safe Haven Transfer

In [107]:
valid_session_ids = []

for session_id in temp_df['Contact Session ID'].unique():
    session_data = temp_df[temp_df['Contact Session ID'] == session_id]
    
    # Find indices where each activity occurs
    callback_indices = session_data[session_data['Activity Name'] == 'CallbackRetry'].index
    language_indices = session_data[session_data['Activity Name'] == 'LanguageSelectionMenu'].index
    
    # Check if LanguageSelectionMenu (last occurrence) comes after CallbackRetry (first occurrence)
    if len(callback_indices) > 0 and len(language_indices) > 0:
        # Check if the LAST LanguageSelectionMenu comes after the FIRST CallbackRetry
        if language_indices.max() > callback_indices.min():
            valid_session_ids.append(session_id)


print(f"Found {len(valid_session_ids)} sessions")

# split valid_session_ids into
# Method: 
# 1. for each session id, find the second occurence of LanguageSelectionMenu in Activity Name
# 2. Take all the indices of that row and after for the rest of that session id, for each session ID value starting from the second LanguageSelectionMenu, append the id with a '1'
df['Modified Contact Session ID'] = df['Contact Session ID'].copy()

for session_id in valid_session_ids:
    # Get all rows for this session
    session_mask = df['Contact Session ID'] == session_id
    session_data = df[session_mask]
    
    # Find all occurrences of LanguageSelectionMenu
    language_menu_indices = session_data[session_data['Activity Name'] == 'LanguageSelectionMenu'].index
    
    # If there's a second occurrence
    if len(language_menu_indices) >= 2:
        second_occurrence_idx = language_menu_indices[1]
        
        # Append '1' to all rows from the second occurrence onwards
        df.loc[session_mask & (df.index >= second_occurrence_idx), 'Modified Contact Session ID'] = session_id + '1'

# Now use Modified Contact Session ID instead of Contact Session ID
df['Contact Session ID'] = df['Modified Contact Session ID']
df.drop('Modified Contact Session ID', axis=1, inplace=True)



Found 11 sessions


In [110]:
# Function to find the correct start index for each session
def find_call_start(group):
    """
    Find index where pattern occurs, excluding 'CBT Agent'
    """
    agent_col = group['Agent Name'].reset_index(drop=True)
    
    for i in range(len(agent_col) - 3):
        current_agent = agent_col.iloc[i]
        
        # Check the pattern (and exclude CBT Agent)
        if (pd.notna(current_agent) and 
            current_agent != 'CBT Agent' and
            (pd.notna(agent_col.iloc[i + 1]) or (pd.notna(agent_col.iloc[i+2]) and pd.notna(agent_col.iloc[i + 3])))):
            
            return group.iloc[i]['Activity Start Timestamp']
    
    return None

# Convert timestamp
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'])

# Find start times for each session using the pattern
start_times = df.groupby('Contact Session ID').apply(find_call_start)
start_times = start_times.dropna()

# Get the last timestamp per session (end time)
end_times = df.groupby('Contact Session ID')['Activity Start Timestamp'].max()

# Get the last timestamp where Agent Name is not NA (end time)
# agent_present = df[df['Agent Name'].notna()].copy()
# end_times = agent_present.groupby('Contact Session ID')['Activity Start Timestamp'].max()

# Only keep sessions that have both start and end times
valid_sessions = start_times.index.intersection(end_times.index)
start_times = start_times[valid_sessions]
end_times = end_times[valid_sessions]

# Calculate differences
time_diff = ((end_times - start_times).dt.total_seconds() / 60).dropna()

print(f"Found {len(time_diff)} valid sessions")
print(f"Mean time: {time_diff.mean():.2f} minutes")
print(f"Median time: {time_diff.median():.2f} minutes")

C:\Users\Owner\AppData\Local\Temp\ipykernel_19860\822755154.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  start_times = df.groupby('Contact Session ID').apply(find_call_start)


Found 17758 valid sessions
Mean time: 8.90 minutes
Median time: 4.61 minutes


Some quick math shows that for 8 hour days we have about 326 work days of call time. Spread among 8 agents thats about 40.7 days of straight calling per person. This feels low